# P1 — Catálogo de 4 Tareas Acotadas (Baseline Tabular)

**Curso:** DS5345 · Aprendizaje por Refuerzo (2026-II)

Este cuaderno entrena **Monte Carlo Control** y **Q-Learning** (ε fijo vs decay) sobre las cuatro tareas del enunciado:

1. Persecución / intercepción de balón
2. Conducción y drible
3. Tiro a puerta (con/sin portero)
4. Cooperación 2v1 y pase

Los entornos viven en `src/envs/`. Para regenerar figuras offline:

```bash
python scripts/run_all_tasks.py
```


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np

from agents.mc_control import train_mc_control
from agents.q_learning import train_q_learning
from envs import TASK_ENVS
from metrics import evaluate_shooting_split, summarize_run
from plotting import plot_learning_curves, plot_trajectory, rollout_greedy

N_EPISODES = 1500  # demo rápida; usar 3500 en scripts/run_all_tasks.py
SEED = 42
CONFIGS = [
    ("MC | ε decay", train_mc_control, {"exploration": "decay"}),
    ("MC | ε fijo=0.1", train_mc_control, {"exploration": "fixed", "eps_fixed": 0.1}),
    ("QL | ε decay", train_q_learning, {"exploration": "decay", "alpha": 0.1}),
    ("QL | ε fijo=0.1", train_q_learning, {"exploration": "fixed", "eps_fixed": 0.1, "alpha": 0.1}),
]
print("Listo. Tareas:", list(TASK_ENVS))


## Entrenamiento por tarea

Para cada tarea se comparan las 4 configuraciones y se grafican curvas + una trayectoria greedy de la mejor política.


In [ ]:
def run_task_demo(task_name: str, n_episodes: int = N_EPISODES):
    EnvCls = TASK_ENVS[task_name]
    results, summaries = {}, {}
    for name, trainer, kwargs in CONFIGS:
        env = EnvCls(seed=SEED)
        res = trainer(env=env, n_episodes=n_episodes, seed=SEED, **kwargs)
        results[name] = res
        summaries[name] = summarize_run(res, last_n=100)
        print(f"{task_name} | {name}: {summaries[name]}")

    plot_learning_curves(results, window=40, title=f"Curvas — {task_name}")
    plt.show()

    best_name = max(summaries, key=lambda k: summaries[k]["success_rate_last"])
    best = results[best_name]
    env_vis = EnvCls(seed=7)
    if task_name == "shooting":
        env_vis = EnvCls(seed=7, with_gk=False)
        split = evaluate_shooting_split(best["Q"], seed=99, n_episodes=150)
        print("Eval tiro abierto / con portero:", split)
    rollout_greedy(env_vis, best["Q"])
    plot_trajectory(env_vis, title=f"Trayectoria greedy — {best_name}")
    plt.show()
    return results, summaries, best_name

all_summaries = {}
for task in ("pursuit", "dribbling", "shooting", "passing"):
    print("\n" + "=" * 60)
    _, summaries, best = run_task_demo(task)
    all_summaries[task] = {"best": best, "summaries": summaries}

print("\nResumen mejores configs:")
for t, info in all_summaries.items():
    s = info["summaries"][info["best"]]
    print(f"  {t}: {info['best']} → éxito={s['success_rate_last']:.0%}")


## Criterios del enunciado (referencia)

| Tarea | Criterio |
|---|---|
| Pursuit | Captura >90% en <40 pasos |
| Dribbling | Avance >30 m con posesión |
| Shooting | >75% arco abierto y >50% con portero |
| Passing | ≥50 pasos de posesión y ≥3 pases |

Las métricas completas a 3500 episodios están en `figures/metrics_all_tasks.json`.
